# Annotate Word documents using the ASE analyzer

Create a highlighted `.docx` from a `.docx` or legacy `.doc`. Text is analyzed through your ASE API, including matches across Word runs, tables, hyperlinks, headers, and footers. Embedded PNG/JPEG pictures receive yellow OCR-based outlines. **Annotation does not remove PII: original document content and metadata remain.**

Copy this notebook **and `ase_document_tools.py`** to the kernel host. Set `TOOLS_DIRECTORY` below to the folder containing the helper. Use the same VM managed identity and private network access as your working ASE notebook; no local Presidio engine or spaCy model is used.

Install Python dependencies on the kernel host:
```powershell
uv pip install --python .venv-ase\Scripts\python.exe requests azure-identity ipykernel Pillow
```

Install **Tesseract OCR plus language data** on the VM for embedded images; install **LibreOffice** for legacy `.doc` conversion. These are native programs, not Python packages. Configure full executable paths if not on PATH. Conversion uses a temporary isolated profile with macros disabled, but only process trusted documents on an updated host. Legacy conversion can change layout and produces DOCX output.

Use Python 3.10 through 3.14. Start with synthetic files. Never display input text, tokens, raw error bodies, or whole document contents. Clear notebook outputs before sharing.


In [ ]:
import os
import sys
from pathlib import Path

TOOLS_DIRECTORY = Path(os.environ.get("ASE_SAMPLE_DIRECTORY", str(Path.cwd())))
ANALYZER_BASE_URL = os.environ.get("ANALYZER_BASE_URL", "https://presidio-analyzer.ase-jtl.appserviceenvironment.us")
ANALYZER_API_RESOURCE = os.environ.get("ANALYZER_API_RESOURCE", "")
MANAGED_IDENTITY_CLIENT_ID = os.environ.get("MANAGED_IDENTITY_CLIENT_ID", "")
INPUT_DOCUMENT = Path(os.environ.get("INPUT_DOCUMENT", str(Path("sample_data") / "sample.docx")))
OUTPUT_DOCUMENT = Path(os.environ.get("OUTPUT_DOCUMENT", str(INPUT_DOCUMENT.with_name(INPUT_DOCUMENT.stem + "_annotated.docx"))))
LANGUAGE = "en"
OCR_LANGUAGE = "eng"  # Tesseract language code, distinct from the analyzer's ISO code.
OCR_MIN_CONFIDENCE = 50
TESSERACT_COMMAND = os.environ.get("TESSERACT_COMMAND", "tesseract")
LIBREOFFICE_COMMAND = os.environ.get("LIBREOFFICE_COMMAND", "soffice")


Set the registered analyzer **Application ID URI**, without `/.default`, and the calling UMI **client ID**. These values are not inherited automatically from another notebook. For your Government v1 token, retain the working App Service issuer `https://login.microsoftonline.us/<tenant-id>/` without `/v2.0`, the tenant restriction, and the caller allowlists. Keep TLS verification enabled; configure `REQUESTS_CA_BUNDLE` if required.

Use absolute input/output paths when the kernel working directory differs from your notebook folder. Choose a new output name in an existing folder; overwriting is refused. Windows executable examples: `C:\Program Files\Tesseract-OCR\tesseract.exe` and `C:\Program Files\LibreOffice\program\soffice.com`.


In [ ]:
if not (TOOLS_DIRECTORY / "ase_document_tools.py").is_file():
    raise FileNotFoundError("Copy ase_document_tools.py to the kernel host and set TOOLS_DIRECTORY to its folder.")
sys.path.insert(0, str(TOOLS_DIRECTORY.resolve()))
from ase_document_tools import ASEAnalyzer, annotate_word

with ASEAnalyzer(ANALYZER_BASE_URL, ANALYZER_API_RESOURCE, MANAGED_IDENTITY_CLIENT_ID, LANGUAGE) as analyzer:
    summary = annotate_word(
        INPUT_DOCUMENT, OUTPUT_DOCUMENT, analyzer.analyze,
        libreoffice=LIBREOFFICE_COMMAND, ocr_language=OCR_LANGUAGE,
        tesseract=TESSERACT_COMMAND, min_confidence=OCR_MIN_CONFIDENCE,
    )
print(f"Saved annotated DOCX: {summary['text_detections']} text detections and {summary['image_detections']} image detections. Original PII remains.")


## Review and limitations

Open the output in an approved viewer on the private host. OCR outlines cover whole detected words, not exact character contours. OCR/detection can miss PII; a zero count is not proof of privacy. Nonuniform images with no OCR text, low-confidence OCR, or invalid geometry stop processing without publishing an output.

Unsupported Word features fail explicitly: multiple sections, footnotes/endnotes, tracked changes, fields (including automatic page numbers/TOCs), text boxes, content controls, equations, floating drawings, vector/multi-frame images, charts/SmartArt, OLE objects, macros, and external linked content except ordinary hyperlinks. Simplify a copy of the document or split it before processing. Annotation retains comments, properties, hyperlink targets, and other original package content; it is **not redaction**.

Authentication errors: check the API resource, VM identity client ID, audience, issuer/token version, tenant, and caller/network restrictions. Native tool errors: install programs/language packs on the kernel host. Never bypass TLS or broaden access permissions to mask a failure.
